This code aims to fit KPFM images line by line into a time series. 
Images must be imported as .npy for correct fitting. 

In [ ]:
#Imports
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import pandas as pd
import scipy.stats as stats 
import matplotlib.colors as mcolors
import matplotlib.lines as mlines
import tkinter as tk
from tkinter import filedialog
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import ListedColormap
import matplotlib.cm as cm
from matplotlib.ticker import MaxNLocator
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d
from tqdm import tqdm

ModuleNotFoundError: No module named 'natsort'

In [ ]:
#Input directory and file saving function 
root = tk.Tk()
root.withdraw()
print("If pop up box not seen minimise spyder and select folder wanted for analysis")
#folder_path = filedialog.askdirectory(title='Select input folder')
save_path = filedialog.askdirectory(title='Select Folder to Save Plot')


def save_plot_with_folder_dialog(fig, default_filename='plot.jpeg'):
    root = tk.Tk()
    root.withdraw()  # Hide the root window

    folder_path = filedialog.askdirectory(title='Select Folder to Save Plot')
    if not folder_path:
        print("Save cancelled.")
        return

    full_path = os.path.join(folder_path, default_filename)
    fig.savefig(full_path)
    print(f"Plot saved to: {full_path}")
    
def save_plot_to_folder(fig, folder_path, filename='plot.jpeg'):
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)  # Create the folder if it doesn't exist
    
    full_path = os.path.join(folder_path, filename)
    fig.tight_layout()
    fig.savefig(full_path, dpi=600, bbox_inches='tight')
    print(f"Plot saved to: {full_path}")

In [ ]:
#Colourmap formatting
# Change c1 and c2 based on the min and max colours to be chosen 

def hex_to_RGB(hex_str):
    
    """ #FFFFFF -> [255,255,255]"""
    
    return [int(hex_str[i:i+2], 16) for i in range(1,6,2)]

def get_color_gradient(c1, c2, n):

    assert n > 1
    c1_rgb = np.array(hex_to_RGB(c1))/255
    c2_rgb = np.array(hex_to_RGB(c2))/255
    mix_pcts = [x/(n-1) for x in range(n)]
    rgb_colors = [((1-mix)*c1_rgb + (mix*c2_rgb)) for mix in mix_pcts]
    return ["#" + "".join([format(int(round(val*255)), "02x") for val in item]) for item in rgb_colors]


colors=[]
c1='#1A85FF'#deep blue
c2='#D41159' # deep pink


GB_U_color = "#5CBBFF"
GI_U_color = "#003357"
GB_P_color = "#D69AAB"
GI_P_color = "#8D1730"

In [ ]:
#Single Gaussian Fitting function: 
def gaussian(x, A, mu, sigma):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2))

In [ ]:
# Gaussian fitting function using data from a dataset (i.e. from each individual file) 
def fit_gaussian_from_dataset(x_data, y_data):
    """
    Fit a Gaussian curve to the provided dataset (x_data, y_data).
    Args:
    - x_data: Array-like, x values of the data
    - y_data: Array-like, y values of the data
    
    Returns:
    - A_fit: Amplitude of the fitted Gaussian
    - mu_fit: Mean (center) of the fitted Gaussian
    - sigma_fit: Standard deviation (width) of the fitted Gaussian
    - popt: Optimal parameters from curve fitting
    - pcov: Covariance matrix
    """
    # Initial guess for the parameters [A, mu, sigma]
    initial_guess = [max(y_data), np.mean(x_data), np.std(x_data)]

    try:
        # Fit the Gaussian curve to the data
        popt, pcov = curve_fit(gaussian, x_data, y_data, p0=initial_guess, maxfev=100000000)
        
        # Extract fitted parameters
        A_fit, mu_fit, sigma_fit = popt
        
        # Return the fitted parameters and covariance matrix
        return A_fit, mu_fit, sigma_fit, popt, pcov

    except Exception as e:
        print(f"Error fitting Gaussian: {e}")
        return None, None, None, None, None

In [ ]:
#Fitting for single gaussian
def fit_gaussian_from_files(file_directory, rows, image_time):
    time_scatter=[]
    gaussian_scatter=[]
    t=-int(image_time)
    gaussian_min=[]
    gaussian_max=[]

    #Sorting files in order
    search_path = os.path.join(file_directory, "*.npy")
    file_list = glob.glob(search_path)
    file_list.sort()
    n_files=len(file_list)
    if not file_list:
        print("No .npy files found in the directory.")
        return

    # Loop through files in the specified directory
 #   for filename in os.listdir(file_directory):
    for idx, filename in enumerate(tqdm(file_list, desc="Processing files")):
        file_path = os.path.join(file_directory, filename)
        #print(f"\nProcessing file: {filename}")
        t += image_time
        time_scatter.append(t)

        with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
            unsorted_data = np.load(f, skiprows=rows)
            data=np.flipud(unsorted_data)

        for i in range(data.shape[0]):
            y_data, voltage = np.histogram(a[i,:], bins='auto')
            popt, pcov = curve_fit(gaussian, voltage[:-1],y_data, maxfev=1000000)
                             
        x_data = data[:, 0]
        y_data = data[:, 1]
            
            # Fit the Gaussian curve to this dataset
        A, mu, sigma, popt, pcov = fit_gaussian_from_dataset(x_data, y_data)

        y_fit=gaussian(x_data,*popt)
        #for finding peak centre for fitted gaussian
        peak_index=np.argmax(y_fit)
        x_fit_max=x_data[peak_index]
        gaussian_scatter.append(x_fit_max)
        centres=[]
        # Finding min/max from sigma's:
        for i in range(len(popt) // 3):
            amp = popt[i*3]
            cen = popt[i*3+1]
            wid = abs(popt[i*3+2])
            centres.append({'amp': amp, 'cen': cen, 'wid': wid})
        
        highest = centres[np.argmax([g['amp'] for g in centres])]
        # FWHM = 2.355 * sigma
        fwhm = 2.355 * highest['wid']
        x_left = highest['cen'] - fwhm / 2
        x_right = highest['cen'] + fwhm / 2

        gaussian_min.append(x_left)
        gaussian_max.append(x_right)


        #Uncomment for debugging: 
        #print(f"Min:{min_scatter}\n Mean: {mean_scatter}\n Max: {max_scatter}")
        #print(time_scatter)
        # If fitting was successful, plot the result
        if A is not None:
                ###PLOTTING PARAMETERS###
            #Pre-setting
            dpi_setting = 600
            figsize_x, figsize_y = (1147/dpi_setting), (770/dpi_setting)
            tick_length = 6
            legend_font = 12
            base_font_size = 12
            scatter_size = 18
            line_width = 3

            #Creating figure and adjusting parameters
            fig, ax = plt.subplots(1,1, sharey = True, figsize = (figsize_x,figsize_y))
            plt.rcParams.update({'font.size': base_font_size})
            ax = plt.gca()
            ax.tick_params(direction='in', length=tick_length)
            ax.tick_params(axis='both', labelsize=base_font_size)
            ax.margins(0.5,0.5)
            #Axis Labels
            plt.xlabel("SPV (V)", size = base_font_size)
            plt.ylabel("Distribution (V$^{-1}$)", fontsize=base_font_size)

            # Formatting ticks
            ax.autoscale(enable=None, axis="x", tight=True)
            ax.xaxis.set_major_locator(MaxNLocator(nbins=5))  # Max 5 ticks on x-axis
            ax.yaxis.set_major_locator(MaxNLocator(nbins=5))  # Max 5 ticks on y-axis

            #Setting colour gradients
            colors=get_color_gradient(c1,c2,n_files)

            plt.scatter(x_data, y_data, label=f"{t}s", color=colors[idx], s=scatter_size)
            plt.plot(x_data, y_fit, label=f"_nolegend_", color=colors[idx], linewidth=line_width)
            plt.autoscale()
                
    print("All gaussians processed. Congrats!")

    # Add the colorbar after plotting
    cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)
    norm = mcolors.Normalize(vmin=np.min(time_scatter), vmax=np.max(time_scatter))
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    fig.tight_layout()
    cbar = fig.colorbar(sm, ax=ax, orientation='vertical')
    cbar.set_label('Time (s)', size=base_font_size)
    cbar.ax.tick_params(labelsize=base_font_size)
    plt.autoscale(enable=True, axis='y', tight=False)
    #Comment out if no file saving required
    fig=plt.gcf()
    save_plot_to_folder(fig, save_path, filename='SingleGaussian_Histogram.jpeg')
    return time_scatter, gaussian_scatter
    plt.show() #Keep after saving figure due to errors. 

In [ ]:
# Fitting all images within a folder: 
def fit_gaussian_for_npy(file_directory, scan_rate, rows):
    image_time=1/scan_rate
    # Data containers 
    time_scatter = []
    gaussian_peaks = []
    gaussian_min = []
    gaussian_max = []
    fwhm_array =  []
    
    # Sorting all files: 
    search_path = os.path.join(file_directory, '*.npy')
    file_list=glob.glob(search_path)
    file_list.sort()
    if not file_list:
            print("No .npy files found.")
            return

    # Setting global plotting parameters: 
    dpi_setting = 600
    figsize_x, figsize_y = (2227/dpi_setting), (1498/dpi_setting) # ~3.7 x 2.5 inches
    base_font_size = 12
    plt.rcParams.update({'font.size': base_font_size})
    # Plotting fit of last line ONLY
    fig, ax = plt.subplots(figsize=(figsize_x, figsize_y))
    #Processing files in order: 

    total_line_counter = 0
    

    for idx, file_path in enumerate(tqdm(file_list, desc="Processing images")):
            # Expected shape (32,64)
            #Flipping images for correct time order: 
            raw_image=np.load(file_path)
            data=np.flipud

            for r in range(data.shape[0]): 
                line_data=data[r,:]
                counts, bin_edges = np.histogram(line_data, bins='auto', density=True)
                bin_centres = (bin_edges[:-1] + bin_edges[1:]) / 2
                total_line_counter=+1
                time_scatter.append(image_time*total_line_counter-image_time)
                try:
                    # Initial guess: [Amplitude, Mean, Sigma]
                    p0 = [np.max(counts), np.mean(line_data), np.std(line_data)]
                    popt, _ = curve_fit(gaussian, bin_centres, counts, p0=p0, maxfev=1000000)
                    
                    # Extract results
                    amp, cen, sigma = popt
                    fwhm = 2.355 * abs(sigma)

                    # Appending to lists: 
                    gaussian_peaks.append(cen)
                    gaussian_min.append(cen - fwhm/2)
                    gaussian_max.append(cen + fwhm/2)
                    # Plotting only the last line as a visual check
                    if idx == len(file_list)-1 and r == data.shape[0]-1:
                        ax.scatter(bin_centres, counts, s=10, alpha=0.5, label="Raw Data (Last Line)")
                        x_fine = np.linspace(bin_centres.min(), bin_centres.max(), 100)
                        ax.plot(x_fine, gaussian(x_fine, *popt), 'r-', linewidth=2, label="Gaussian Fit")
                except Exception as e:
                    # If a specific line fails to fit, we skip it to keep the code running
                    continue

    #Final Plotting: 
    ax.set_xlabel("SPV (V)")
    ax.set_ylabel("Distribution (V$^{-1}$)")
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5))    
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))    
    ax.tick_params(direction='in', length=6, top=True, right=True)
    plt.tight_layout()
    plt.show()
    print(f"Processed {len(gaussian_peaks)} lines successfully.")
    return time_scatter, gaussian_peaks, gaussian_min,gaussian_max
                            

In [ ]:
rows = 3
scan_rate=0.75 #Hz

In [ ]:
Unpass_On = filedialog.askdirectory(title='Select input folder')
UnpassOn_Time, UnpassOn_peaks, UnpassOn_min,UnpassOn_max= fit_gaussian_for_npy(Unpass_On, scan_rate, rows)






In [ ]:
Unpass_Off = filedialog.askdirectory(title='Select input folder')

In [ ]:
Pass_On = filedialog.askdirectory(title='Select input folder')

In [ ]:
Pass_Off = filedialog.askdirectory(title='Select input folder')

In [ ]:
save_path = filedialog.askdirectory(title='Select input folder')